In [1]:
from pathlib import Path

from bm25_csv_evaluator import (
    BM25EvalConfig,
    print_evaluation_report,
    run_bm25_csv_evaluation,
)

CSV_PATH = "valset_synthetic.csv"
THEOREM_TABLE = "theorem"
SLOGAN_TABLE = "theorem_slogan"
SLOGAN_ID_COL = "slogan_id"
SLOGAN_TEXT_COL = "slogan"
THEOREM_ID_COL = "theorem_id"
THEOREM_NAME_COL = "name"
THEOREM_PAPER_ID_COL = "paper_id"
BATCH_SIZE = 5000
LIMIT = None
K1 = 1.5
B = 0.75
RAW_TOP_K = 200
EVAL_TOP_K = 20
MAX_EXAMPLE_MISSES = 5

In [ ]:
# First experiment: BM25 only
config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=SLOGAN_TABLE,
    slogan_id_col=SLOGAN_ID_COL,
    slogan_text_col=SLOGAN_TEXT_COL,
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=EVAL_TOP_K,
)

evaluation = run_bm25_csv_evaluation(config)
print_evaluation_report(evaluation, max_examples=MAX_EXAMPLE_MISSES)

evaluation["metrics"]


In [ ]:
# Second experiment: BM25 on LaTeX
body_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=THEOREM_TABLE,
    slogan_id_col=THEOREM_ID_COL,
    slogan_text_col="body",
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=EVAL_TOP_K,
)

body_evaluation = run_bm25_csv_evaluation(body_config)
print_evaluation_report(body_evaluation, max_examples=MAX_EXAMPLE_MISSES)

body_evaluation["metrics"]


In [ ]:
# Third experiment: Reciprocal Rank Fusion (RRF) of BM25 with dense API search
import importlib
from pathlib import Path

from bm25_csv_evaluator import BM25EvalConfig
import rrf_csv_evaluator

importlib.reload(rrf_csv_evaluator)

from rrf_csv_evaluator import (
    RRFExperimentConfig,
    print_rrf_evaluation_report,
    run_rrf_csv_evaluation,
)

RRF_DENSE_SEARCH_URL = "https://api.theoremsearch.com/search"
RRF_DENSE_N_RESULTS = EVAL_TOP_K
RRF_DENSE_MIN_N_RESULTS = EVAL_TOP_K
RRF_QUERY_DELAY_SECONDS = 4.0
RRF_K = 60
RRF_TIMEOUT_SECONDS = 60

# Use a deeper BM25 candidate list for fusion, then evaluate the fused top-20.
rrf_bm25_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=SLOGAN_TABLE,
    slogan_id_col=SLOGAN_ID_COL,
    slogan_text_col=SLOGAN_TEXT_COL,
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=RAW_TOP_K,
)

rrf_config = RRFExperimentConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    bm25_config=rrf_bm25_config,
    dense_search_url=RRF_DENSE_SEARCH_URL,
    dense_n_results=RRF_DENSE_N_RESULTS,
    dense_min_n_results=RRF_DENSE_MIN_N_RESULTS,
    inter_query_delay_seconds=RRF_QUERY_DELAY_SECONDS,
    rrf_k=RRF_K,
    eval_top_k=EVAL_TOP_K,
    timeout_seconds=RRF_TIMEOUT_SECONDS,
    dense_max_retries=2,
    dense_retry_backoff_seconds=1.0,
    continue_on_dense_error=True,
)

rrf_evaluation = run_rrf_csv_evaluation(rrf_config)
print_rrf_evaluation_report(rrf_evaluation, max_examples=MAX_EXAMPLE_MISSES)

rrf_evaluation["metrics"]


In [ ]:
# Fourth experiment: dense search only
import importlib
from pathlib import Path

from bm25_csv_evaluator import BM25EvalConfig
import rrf_csv_evaluator

importlib.reload(rrf_csv_evaluator)

from rrf_csv_evaluator import (
    RRFExperimentConfig,
    print_dense_evaluation_report,
    run_dense_csv_evaluation,
)

DENSE_ONLY_SEARCH_URL = "https://api.theoremsearch.com/search"
DENSE_ONLY_N_RESULTS = EVAL_TOP_K
DENSE_ONLY_MIN_N_RESULTS = EVAL_TOP_K
DENSE_ONLY_QUERY_DELAY_SECONDS = 4.0
DENSE_ONLY_TIMEOUT_SECONDS = 60

# Reuse theorem metadata config for resolving dense API results.
dense_only_meta_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=SLOGAN_TABLE,
    slogan_id_col=SLOGAN_ID_COL,
    slogan_text_col=SLOGAN_TEXT_COL,
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=RAW_TOP_K,
)

dense_only_config = RRFExperimentConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    bm25_config=dense_only_meta_config,
    dense_search_url=DENSE_ONLY_SEARCH_URL,
    dense_n_results=DENSE_ONLY_N_RESULTS,
    dense_min_n_results=DENSE_ONLY_MIN_N_RESULTS,
    inter_query_delay_seconds=DENSE_ONLY_QUERY_DELAY_SECONDS,
    eval_top_k=EVAL_TOP_K,
    timeout_seconds=DENSE_ONLY_TIMEOUT_SECONDS,
    dense_max_retries=2,
    dense_retry_backoff_seconds=1.0,
    continue_on_dense_error=True,
)

dense_only_evaluation = run_dense_csv_evaluation(dense_only_config)
print_dense_evaluation_report(dense_only_evaluation, max_examples=MAX_EXAMPLE_MISSES)

dense_only_evaluation["metrics"]


In [ ]:
# Fifth experiment: Reciprocal Rank Fusion (RRF) of BM25 over theorem.body with local Gemma pgvector search
import importlib
from pathlib import Path

from bm25_csv_evaluator import BM25EvalConfig
import rrf_csv_evaluator

importlib.reload(rrf_csv_evaluator)

from rrf_csv_evaluator import (
    RRFExperimentConfig,
    print_rrf_evaluation_report,
    run_local_rrf_csv_evaluation,
)

LOCAL_GEMMA_VECTOR_TABLE = "raw_theorem_embedding_gemma"
LOCAL_GEMMA_VECTOR_COL = "embedding"
LOCAL_GEMMA_MODEL_NAME = "google/embeddinggemma-300m"
LOCAL_GEMMA_PROMPT = (
    "Instruct: Given a math search query, retrieve theorems "
    "mathematically equivalent to the query.\nQuery:"
)
LOCAL_GEMMA_N_RESULTS = EVAL_TOP_K
LOCAL_GEMMA_QUERY_DELAY_SECONDS = 0.0
LOCAL_GEMMA_RRF_K = 60

# Reuse the body BM25 setup from experiment 2, but keep a deeper BM25 list for fusion.
local_gemma_body_bm25_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=THEOREM_TABLE,
    slogan_id_col=THEOREM_ID_COL,
    slogan_text_col="body",
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=RAW_TOP_K,
)

local_gemma_rrf_config = RRFExperimentConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    bm25_config=local_gemma_body_bm25_config,
    dense_n_results=LOCAL_GEMMA_N_RESULTS,
    dense_min_n_results=LOCAL_GEMMA_N_RESULTS,
    inter_query_delay_seconds=LOCAL_GEMMA_QUERY_DELAY_SECONDS,
    rrf_k=LOCAL_GEMMA_RRF_K,
    eval_top_k=EVAL_TOP_K,
    continue_on_dense_error=False,
    local_dense_table=LOCAL_GEMMA_VECTOR_TABLE,
    local_dense_join_col=THEOREM_ID_COL,
    local_dense_vector_col=LOCAL_GEMMA_VECTOR_COL,
    local_dense_metric="cosine",
    local_embedding_model_name=LOCAL_GEMMA_MODEL_NAME,
    local_embedding_prompt=LOCAL_GEMMA_PROMPT,
    local_embedding_device="cpu",
)

local_gemma_rrf_evaluation = run_local_rrf_csv_evaluation(local_gemma_rrf_config)
print_rrf_evaluation_report(local_gemma_rrf_evaluation, max_examples=MAX_EXAMPLE_MISSES)

local_gemma_rrf_evaluation["metrics"]


Loaded 100,000 slogans...
Loaded 200,000 slogans...
Loaded 300,000 slogans...
Loaded 400,000 slogans...
Loaded 500,000 slogans...
Loaded 600,000 slogans...
Loaded 700,000 slogans...
Loaded 800,000 slogans...
Loaded 900,000 slogans...
Loaded 1,000,000 slogans...
Loaded 1,100,000 slogans...
Loaded 1,200,000 slogans...
Loaded 1,300,000 slogans...
Loaded 1,400,000 slogans...
Loaded 1,500,000 slogans...
Loaded 1,600,000 slogans...
Loaded 1,700,000 slogans...
Loaded 1,800,000 slogans...
Loaded 1,900,000 slogans...
Loaded 2,000,000 slogans...
Loaded 2,100,000 slogans...
Loaded 2,200,000 slogans...
Loaded 2,300,000 slogans...
Loaded 2,400,000 slogans...
Loaded 2,500,000 slogans...
Loaded 2,600,000 slogans...
Loaded 2,700,000 slogans...
Loaded 2,800,000 slogans...
Loaded 2,900,000 slogans...
Loaded 3,000,000 slogans...
Loaded 3,100,000 slogans...
Loaded 3,200,000 slogans...
Loaded 3,300,000 slogans...
Loaded 3,400,000 slogans...
Loaded 3,500,000 slogans...
Loaded 3,600,000 slogans...
Loaded 3,7

/home/lukealex/code/TheoremSearch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local dense model 'google/embeddinggemma-300m' on 'cpu'...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 314/314 [00:00<00:00, 3586.55it/s]


[1/299] Local Gemma RRF match position: miss


In [ ]:
print("Second experiment")

# Second experiment: BM25 on LaTeX
body_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=THEOREM_TABLE,
    slogan_id_col=THEOREM_ID_COL,
    slogan_text_col="body",
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=EVAL_TOP_K,
)

body_evaluation = run_bm25_csv_evaluation(body_config)
print_evaluation_report(body_evaluation, max_examples=MAX_EXAMPLE_MISSES)

body_evaluation["metrics"]

print("Third experiment")

# Third experiment: Reciprocal Rank Fusion (RRF) of BM25 with dense API search
import importlib
from pathlib import Path

from bm25_csv_evaluator import BM25EvalConfig
import rrf_csv_evaluator

importlib.reload(rrf_csv_evaluator)

from rrf_csv_evaluator import (
    RRFExperimentConfig,
    print_rrf_evaluation_report,
    run_rrf_csv_evaluation,
)

RRF_DENSE_SEARCH_URL = "https://api.theoremsearch.com/search"
RRF_DENSE_N_RESULTS = EVAL_TOP_K
RRF_DENSE_MIN_N_RESULTS = EVAL_TOP_K
RRF_QUERY_DELAY_SECONDS = 4.0
RRF_K = 60
RRF_TIMEOUT_SECONDS = 60

# Use a deeper BM25 candidate list for fusion, then evaluate the fused top-20.
rrf_bm25_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=SLOGAN_TABLE,
    slogan_id_col=SLOGAN_ID_COL,
    slogan_text_col=SLOGAN_TEXT_COL,
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=RAW_TOP_K,
)

rrf_config = RRFExperimentConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    bm25_config=rrf_bm25_config,
    dense_search_url=RRF_DENSE_SEARCH_URL,
    dense_n_results=RRF_DENSE_N_RESULTS,
    dense_min_n_results=RRF_DENSE_MIN_N_RESULTS,
    inter_query_delay_seconds=RRF_QUERY_DELAY_SECONDS,
    rrf_k=RRF_K,
    eval_top_k=EVAL_TOP_K,
    timeout_seconds=RRF_TIMEOUT_SECONDS,
    dense_max_retries=2,
    dense_retry_backoff_seconds=1.0,
    continue_on_dense_error=True,
)

rrf_evaluation = run_rrf_csv_evaluation(rrf_config)
print_rrf_evaluation_report(rrf_evaluation, max_examples=MAX_EXAMPLE_MISSES)

rrf_evaluation["metrics"]

print("fifth experiment")
# Fifth experiment: Reciprocal Rank Fusion (RRF) of BM25 over theorem.body with local Gemma pgvector search
import importlib
from pathlib import Path

from bm25_csv_evaluator import BM25EvalConfig
import rrf_csv_evaluator

importlib.reload(rrf_csv_evaluator)

from rrf_csv_evaluator import (
    RRFExperimentConfig,
    print_rrf_evaluation_report,
    run_local_rrf_csv_evaluation,
)

LOCAL_GEMMA_VECTOR_TABLE = "raw_theorem_embedding_gemma"
LOCAL_GEMMA_VECTOR_COL = "embedding"
LOCAL_GEMMA_MODEL_NAME = "google/embeddinggemma-300m"
LOCAL_GEMMA_PROMPT = (
    "Instruct: Given a math search query, retrieve theorems "
    "mathematically equivalent to the query.\nQuery:"
)
LOCAL_GEMMA_N_RESULTS = EVAL_TOP_K
LOCAL_GEMMA_QUERY_DELAY_SECONDS = 0.0
LOCAL_GEMMA_RRF_K = 60

# Reuse the body BM25 setup from experiment 2, but keep a deeper BM25 list for fusion.
local_gemma_body_bm25_config = BM25EvalConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    theorem_table=THEOREM_TABLE,
    slogan_table=THEOREM_TABLE,
    slogan_id_col=THEOREM_ID_COL,
    slogan_text_col="body",
    theorem_id_col=THEOREM_ID_COL,
    theorem_name_col=THEOREM_NAME_COL,
    theorem_paper_id_col=THEOREM_PAPER_ID_COL,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    k1=K1,
    b=B,
    raw_top_k=RAW_TOP_K,
    eval_top_k=RAW_TOP_K,
)

local_gemma_rrf_config = RRFExperimentConfig(
    csv_path=str(Path(CSV_PATH).expanduser()),
    bm25_config=local_gemma_body_bm25_config,
    dense_n_results=LOCAL_GEMMA_N_RESULTS,
    dense_min_n_results=LOCAL_GEMMA_N_RESULTS,
    inter_query_delay_seconds=LOCAL_GEMMA_QUERY_DELAY_SECONDS,
    rrf_k=LOCAL_GEMMA_RRF_K,
    eval_top_k=EVAL_TOP_K,
    continue_on_dense_error=False,
    local_dense_table=LOCAL_GEMMA_VECTOR_TABLE,
    local_dense_join_col=THEOREM_ID_COL,
    local_dense_vector_col=LOCAL_GEMMA_VECTOR_COL,
    local_dense_metric="cosine",
    local_embedding_model_name=LOCAL_GEMMA_MODEL_NAME,
    local_embedding_prompt=LOCAL_GEMMA_PROMPT,
    local_embedding_device="cpu",
)

local_gemma_rrf_evaluation = run_local_rrf_csv_evaluation(local_gemma_rrf_config)
print_rrf_evaluation_report(local_gemma_rrf_evaluation, max_examples=MAX_EXAMPLE_MISSES)

local_gemma_rrf_evaluation["metrics"]

